# Entraînement du Modèle - Dataset Clean_2

Ce notebook entraîne et compare deux modèles sur le nouveau dataset `openfoodfacts_clean_2.csv`:
- **XGBoost Classifier**
- **Random Forest Classifier**

## 1. Imports et Configuration

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(str(Path.cwd().parent))

from src.data_loader import load_data, get_features_and_target
from src.preprocessing import NutriscorePreprocessor
from src.feature_engineering import create_engineered_features
from src.models import create_xgboost_model, create_random_forest_model
from src.training import train_model
from src.evaluation import evaluate_model
from src.prediction import save_model

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Imports réussis")

## 2. Chargement des Données - Clean_2

In [ ]:
data_path = Path.cwd().parent / 'data' / 'openfoodfacts_clean_2.csv'
print(f"Chargement depuis: {data_path}")

df = load_data(str(data_path))
print(f"\n📊 Dataset chargé: {df.shape[0]:,} produits, {df.shape[1]} colonnes")

df.head(10)

In [ ]:
X, y = get_features_and_target(df)

print(f"Features: {X.shape[1]} colonnes")
print(f"\nDistribution du Nutri-Score:")
print(y.value_counts().sort_index())

plt.figure(figsize=(10, 6))
y.value_counts().sort_index().plot(kind='bar', color=['green', 'lightgreen', 'yellow', 'orange', 'red'])
plt.title('Distribution du Nutri-Score - Dataset Clean_2', fontsize=14, fontweight='bold')
plt.xlabel('Grade Nutri-Score')
plt.ylabel('Nombre de produits')
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Feature Engineering

In [ ]:
print("🔧 Application du Feature Engineering...")
X_engineered = create_engineered_features(X)

print(f"\nFeatures originales: {X.shape[1]}")
print(f"Features après engineering: {X_engineered.shape[1]}")
print(f"Nouvelles features créées: {X_engineered.shape[1] - X.shape[1]}")

print("\nListe des features:")
for i, col in enumerate(X_engineered.columns, 1):
    marker = "🆕" if col not in X.columns else "📊"
    print(f"{i:2d}. {marker} {col}")

## 4. Preprocessing et Split

In [ ]:
print("⚙️ Preprocessing des données...")
preprocessor = NutriscorePreprocessor()
X_train, X_test, y_train, y_test = preprocessor.fit_transform(
    X_engineered, y, test_size=0.3, random_state=42
)

print(f"\n✅ Split terminé:")
print(f"   Train set: {X_train.shape[0]:,} échantillons ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"   Test set:  {X_test.shape[0]:,} échantillons ({X_test.shape[0]/len(X)*100:.1f}%)")

print(f"\n📊 Distribution dans le train set:")
train_dist = pd.Series(y_train).value_counts().sort_index()
for grade, count in train_dist.items():
    grade_name = preprocessor.label_encoder.inverse_transform([grade])[0]
    print(f"   Grade {grade_name}: {count:,} ({count/len(y_train)*100:.1f}%)")

## 5. Entraînement des Modèles

### 5.1 XGBoost

In [ ]:
print("🚀 Entraînement XGBoost...\n")
xgb_model = create_xgboost_model()
xgb_model, xgb_metrics = train_model(
    xgb_model, X_train, y_train, X_test, y_test, use_sample_weights=True
)

print("\n✅ XGBoost entraîné!")

### 5.2 Random Forest

In [ ]:
print("🌲 Entraînement Random Forest...\n")
rf_model = create_random_forest_model()
rf_model, rf_metrics = train_model(
    rf_model, X_train, y_train, X_test, y_test, use_sample_weights=False
)

print("\n✅ Random Forest entraîné!")

## 6. Comparaison des Modèles

In [ ]:
comparison_df = pd.DataFrame({
    'Modèle': ['XGBoost', 'Random Forest'],
    'Accuracy Train': [xgb_metrics['train_accuracy'], rf_metrics['train_accuracy']],
    'Accuracy Test': [xgb_metrics['test_accuracy'], rf_metrics['test_accuracy']],
    'F1-Score Train': [xgb_metrics['train_f1'], rf_metrics['train_f1']],
    'F1-Score Test': [xgb_metrics['test_f1'], rf_metrics['test_f1']],
    'Temps (s)': [xgb_metrics['training_time'], rf_metrics['training_time']]
})

print("📊 COMPARAISON - DATASET CLEAN_2\n")
print(comparison_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metrics = ['Accuracy Test', 'F1-Score Test']
x = np.arange(len(metrics))
width = 0.35

axes[0].bar(x - width/2, [xgb_metrics['test_accuracy'], xgb_metrics['test_f1']], 
            width, label='XGBoost', color='steelblue')
axes[0].bar(x + width/2, [rf_metrics['test_accuracy'], rf_metrics['test_f1']], 
            width, label='Random Forest', color='forestgreen')
axes[0].set_ylabel('Score')
axes[0].set_title('Performance - Clean_2', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(['XGBoost', 'Random Forest'], 
            [xgb_metrics['training_time'], rf_metrics['training_time']],
            color=['steelblue', 'forestgreen'])
axes[1].set_ylabel('Temps (secondes)')
axes[1].set_title('Temps d\'Entraînement', fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

best_model = xgb_model if xgb_metrics['test_f1'] > rf_metrics['test_f1'] else rf_model
best_name = 'XGBoost' if xgb_metrics['test_f1'] > rf_metrics['test_f1'] else 'Random Forest'
print(f"\n🏆 Meilleur modèle: {best_name}")

## 7. Évaluation Détaillée

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

y_pred = best_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average='macro')
f1_weighted = f1_score(y_test, y_pred, average='weighted')
accuracy_tolerance = np.mean(np.abs(y_test - y_pred) <= 1)

print(f"📊 ÉVALUATION - {best_name} - CLEAN_2\n")
print(f"Accuracy globale:       {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"F1-Score (macro):       {f1_macro:.4f}")
print(f"F1-Score (weighted):    {f1_weighted:.4f}")
print(f"Accuracy ±1 grade:      {accuracy_tolerance:.4f} ({accuracy_tolerance*100:.2f}%)")

print("\n" + "="*60)
print("RAPPORT DE CLASSIFICATION")
print("="*60)
target_names = preprocessor.label_encoder.classes_
print(classification_report(y_test, y_pred, target_names=target_names))

## 8. Matrice de Confusion

In [ ]:
cm = confusion_matrix(y_test, y_pred)
target_names = preprocessor.label_encoder.classes_

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=target_names, yticklabels=target_names,
            cbar_kws={'label': 'Nombre de prédictions'})
plt.title(f'Matrice de Confusion - {best_name}\nDataset Clean_2', 
          fontsize=14, fontweight='bold')
plt.ylabel('Vraie classe', fontsize=12)
plt.xlabel('Classe prédite', fontsize=12)
plt.tight_layout()
plt.show()

## 9. Importance des Features

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    feature_names = X_train.columns
    
    feature_importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances
    }).sort_values('Importance', ascending=False)
    
    print("🔍 TOP 15 FEATURES - CLEAN_2\n")
    print(feature_importance_df.head(15).to_string(index=False))
    
    plt.figure(figsize=(12, 6))
    top_features = feature_importance_df.head(15)
    plt.barh(range(len(top_features)), top_features['Importance'], color='steelblue')
    plt.yticks(range(len(top_features)), top_features['Feature'])
    plt.xlabel('Importance', fontsize=12)
    plt.title(f'Top 15 Features - {best_name} - Clean_2', fontsize=14, fontweight='bold')
    plt.gca().invert_yaxis()
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

## 10. Sauvegarde du Modèle

In [ ]:
models_dir = Path.cwd().parent / 'models'
models_dir.mkdir(exist_ok=True)

model_path = models_dir / 'nutriscore_model_clean2.pkl'
save_model(best_model, preprocessor, preprocessor.label_encoder, str(model_path))

print(f"\n✅ Modèle sauvegardé!")
print(f"📁 Emplacement: {model_path}")
print(f"📊 Modèle: {best_name}")
print(f"🎯 Accuracy: {accuracy:.4f}")
print(f"📈 F1-Score: {f1_macro:.4f}")

## 11. Résumé Final

In [ ]:
print("="*60)
print("RÉSUMÉ - DATASET CLEAN_2")
print("="*60)
print(f"\n📊 Dataset: openfoodfacts_clean_2.csv")
print(f"   - Total: {len(df):,} produits")
print(f"   - Train: {len(X_train):,} produits")
print(f"   - Test:  {len(X_test):,} produits")
print(f"\n🔧 Features: {X_engineered.shape[1]}")
print(f"\n🏆 Meilleur modèle: {best_name}")
print(f"   - Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"   - F1-Score (macro): {f1_macro:.4f}")
print(f"   - Accuracy ±1 grade: {accuracy_tolerance:.4f} ({accuracy_tolerance*100:.2f}%)")
print(f"\n💾 Modèle sauvegardé: {model_path}")
print("\n✅ Entraînement terminé!")
print("="*60)